In [2]:
import pandas as pd
import numpy as np

# Load original dataset
df = pd.read_csv("../data/raw/nirf_2017_2025_raw.csv")

# Create a separate working copy
clean_df = df.copy()

print("Original dataset shape:", df.shape)
print("Working dataset shape:", clean_df.shape)

# Confirm that the original dataset remains unchanged
print("Both datasets have identical values:",
      df.equals(clean_df))

Original dataset shape: (6966, 13)
Working dataset shape: (6966, 13)
Both datasets have identical values: True


In [3]:
# Create numerical versions without changing original columns

clean_df["Score_numeric"] = pd.to_numeric(
    clean_df["Score"],
    errors="coerce"
)

clean_df["Rank_numeric"] = pd.to_numeric(
    clean_df["Rank"],
    errors="coerce"
)

# Display original and converted values
display(
    clean_df[
        ["Score", "Score_numeric", "Rank", "Rank_numeric"]
    ].head(15)
)

# Check conversion results
print("Numeric Score values:",
      clean_df["Score_numeric"].notna().sum())

print("Missing or non-numeric Score values:",
      clean_df["Score_numeric"].isna().sum())

print("Numeric Rank values:",
      clean_df["Rank_numeric"].notna().sum())

print("Missing or non-numeric Rank values:",
      clean_df["Rank_numeric"].isna().sum())

,Score,Score_numeric,Rank,Rank_numeric
0,83.28,83.28,1,1.0
1,73.97,73.97,2,2.0
2,71.78,71.78,3,3.0
3,68.43,68.43,4,4.0
4,64.18,64.18,5,5.0
5,61.53,61.53,6,6.0
6,60.69,60.69,7,7.0
7,60.37,60.37,8,8.0
8,59.84,59.84,9,9.0
9,58.92,58.92,10,10.0


Numeric Score values: 6923
Missing or non-numeric Score values: 43
Numeric Rank values: 6962
Missing or non-numeric Rank values: 4


In [4]:
# Check whether original columns are preserved

print("Original Score column:")
print(clean_df["Score"].dtype)

print("\nConverted Score column:")
print(clean_df["Score_numeric"].dtype)

print("\nOriginal Rank column:")
print(clean_df["Rank"].dtype)

print("\nConverted Rank column:")
print(clean_df["Rank_numeric"].dtype)

# Confirm original values remain unchanged
print(
    "\nOriginal Score values preserved:",
    clean_df["Score"].equals(df["Score"])
)

print(
    "Original Rank values preserved:",
    clean_df["Rank"].equals(df["Rank"])
)

Original Score column:
object

Converted Score column:
float64

Original Rank column:
object

Converted Rank column:
float64

Original Score values preserved: True
Original Rank values preserved: True


In [5]:
# Count unique institution names and IDs

print("Unique institution names:",
      clean_df["Name"].nunique())

print("Unique institution IDs:",
      clean_df["Institute_ID"].nunique())

# Find IDs associated with multiple names
name_variations = (
    clean_df.groupby("Institute_ID")["Name"]
    .nunique()
    .reset_index(name="Name_Count")
)

name_variations = name_variations[
    name_variations["Name_Count"] > 1
]

print("Institute IDs with multiple names:",
      len(name_variations))

display(name_variations.head(20))

Unique institution names: 1334
Unique institution IDs: 2173
Institute IDs with multiple names: 382


,Institute_ID,Name_Count
478,IR-A-C-29341,2
479,IR-A-C-43708,3
480,IR-A-C-46330,4
483,IR-A-C-57952,2
485,IR-A-U-0072,2
486,IR-A-U-0092,2
488,IR-A-U-0108,3
489,IR-A-U-0116,3
492,IR-A-U-0189,2
496,IR-A-U-0263,2


In [6]:
# Display all category names
categories = sorted(clean_df["Category"].dropna().unique())

print("Total unique categories:", len(categories))

for category in categories:
    print(category)

Total unique categories: 17
Agriculture
Architecture
College
Dental
Engineering
Innovation
Law
Management
Medical
OPENUNIVERSITY
Overall
Pharmacy
Research
SDGInstitutions
SKILLUNIVERSITY
STATEPUBLICUNIVERSITY
University


In [7]:
print("Unique states:", clean_df["State"].nunique())

print("\nState names:")
display(
    clean_df["State"]
    .value_counts(dropna=False)
    .to_frame("Institution_Count")
)

Unique states: 34

State names:


,Institution_Count
State,
Tamil Nadu,1293
Maharashtra,676
Delhi,668
Karnataka,581
Uttar Pradesh,497
Punjab,331
Kerala,329
West Bengal,327
Andhra Pradesh,258


In [8]:
# Select the first 10 IDs having multiple names
sample_ids = name_variations["Institute_ID"].head(10)

# Display all records for these IDs
sample_variations = (
    clean_df[clean_df["Institute_ID"].isin(sample_ids)]
    [["Institute_ID", "Year", "Category", "Name"]]
    .sort_values(["Institute_ID", "Year"])
)

display(sample_variations)

,Institute_ID,Year,Category,Name
5061,IR-A-C-29341,2023,Architecture,Chandigarh College of Architecture Sector-12
5912,IR-A-C-29341,2024,Architecture,"Chandigarh College of Architecture, Sector-12"
1744,IR-A-C-43708,2019,Architecture,College of Engineering
2506,IR-A-C-43708,2020,Architecture,College of Engineering Trivandrum
3346,IR-A-C-43708,2021,Architecture,College of Engineering Trivandrum
4246,IR-A-C-43708,2022,Architecture,"College of Engineering, Trivandrum"
5049,IR-A-C-43708,2023,Architecture,College of Engineering Trivandrum
5900,IR-A-C-43708,2024,Architecture,"College of Engineering, Trivandrum"
6824,IR-A-C-43708,2025,Architecture,College of Engineering Trivandrum
2507,IR-A-C-46330,2020,Architecture,Manipal Academy of Higher Education


In [9]:
# Find institution names associated with multiple IDs

id_variations = (
    clean_df.groupby("Name")["Institute_ID"]
    .nunique()
    .reset_index(name="ID_Count")
)

id_variations = id_variations[
    id_variations["ID_Count"] > 1
].sort_values("ID_Count", ascending=False)

print("Names associated with multiple IDs:",
      len(id_variations))

display(id_variations.head(20))

Names associated with multiple IDs: 543


,Name,ID_Count
586,Jamia Millia Islamia,15
497,Indian Institute of Technology Kharagpur,15
629,Kalinga Institute of Industrial Technology,14
502,Indian Institute of Technology Roorkee,14
943,Panjab University,13
64,Anna University,13
105,Banaras Hindu University,13
28,Aligarh Muslim University,13
47,Amity University,12
488,Indian Institute of Technology Delhi,12


In [10]:
# Display state names after removing leading/trailing spaces
state_check = clean_df["State"].dropna().str.strip()

print("Unique states before standardization:",
      clean_df["State"].nunique())

print("Unique states after trimming spaces:",
      state_check.nunique())

# Display state names in alphabetical order
display(
    pd.DataFrame({
        "Original_State": sorted(clean_df["State"].dropna().unique())
    })
)

Unique states before standardization: 34
Unique states after trimming spaces: 34


,Original_State
0,Andaman and Nicobar Islands
1,Andhra Pradesh
2,Arunachal Pradesh
3,Assam
4,Bihar
5,Chandigarh
6,Chhattisgarh
7,Delhi
8,Goa
9,Gujarat


In [11]:

# Identify state names that differ only in capitalization

state_variations = (
    clean_df.groupby(clean_df["State"].str.strip().str.lower())["State"]
    .unique()
)

# Display groups having more than one spelling
for key, values in state_variations.items():
    if len(values) > 1:
        print("Possible state-name variation:", list(values))


Possible state-name variation: ['Jammu And Kashmir', 'Jammu and Kashmir']


In [12]:

# Create a standardized state column
clean_df["State_Unified"] = clean_df["State"].str.strip()

# Correct the identified capitalization variation
clean_df["State_Unified"] = clean_df["State_Unified"].replace({
    "Jammu And Kashmir": "Jammu and Kashmir"
})

# Validate the changes
print("Original unique states:", clean_df["State"].nunique())

print("Standardized unique states:",
      clean_df["State_Unified"].nunique())

# Check Jammu and Kashmir records
display(
    clean_df[
        clean_df["State_Unified"] == "Jammu and Kashmir"
    ][["Year", "Name", "State", "State_Unified"]]
)


Original unique states: 34
Standardized unique states: 33


,Year,Name,State,State_Unified
162,2017,University of Jammu,Jammu And Kashmir,Jammu and Kashmir
172,2017,University of Kashmir,Jammu And Kashmir,Jammu and Kashmir
570,2018,University of Kashmir,Jammu and Kashmir,Jammu and Kashmir
576,2018,University of Jammu,Jammu and Kashmir,Jammu and Kashmir
646,2018,University of Kashmir,Jammu and Kashmir,Jammu and Kashmir
...,...,...,...,...
6856,2025,Sher-e-Kashmir University of Agricultural Scie...,Jammu and Kashmir,Jammu and Kashmir
6872,2025,Sher-e-Kashmir University of Agricultural Scie...,Jammu and Kashmir,Jammu and Kashmir
6913,2025,University of Kashmir,Jammu and Kashmir,Jammu and Kashmir
6926,2025,University of Jammu,Jammu and Kashmir,Jammu and Kashmir


In [13]:

# Check unique city names

print("Unique cities before standardization:",
      clean_df["City"].nunique())

# Remove leading and trailing spaces for comparison
city_check = clean_df["City"].dropna().str.strip()

print("Unique cities after trimming spaces:",
      city_check.nunique())

# Display the 30 most frequently occurring cities
display(
    clean_df["City"]
    .value_counts(dropna=False)
    .head(30)
    .to_frame("Institution_Count")
)


Unique cities before standardization: 378
Unique cities after trimming spaces: 378


,Institution_Count
City,
Chennai,495
New Delhi,428
Mumbai,282
Coimbatore,251
Bengaluru,232
Pune,228
Delhi,225
Hyderabad,187
Kolkata,161


In [14]:

# Identify city names with different capitalization or spacing
# within the same standardized state

city_variations = (
    clean_df.dropna(subset=["City", "State_Unified"])
    .groupby([
        clean_df["State_Unified"].str.strip().str.lower(),
        clean_df["City"].str.strip().str.lower()
    ])["City"]
    .unique()
)

for key, values in city_variations.items():
    if len(values) > 1:
        print("Possible city variation:", list(values))


Possible city variation: ['YERPEDU', 'Yerpedu']
Possible city variation: ['Bilaspur (Chhattisgarh)', 'BILASPUR (Chhattisgarh)']
Possible city variation: ['New Delhi', 'NEW DELHI']
Possible city variation: ['Surat', 'SURAT']
Possible city variation: ['Nagpur', 'NAGPUR']
Possible city variation: ['Bhubaneswar', 'BHUBANESWAR']
Possible city variation: ['Puducherry', 'PUDUCHERRY']
Possible city variation: ['Hyderabad', 'HYDERABAD']
Possible city variation: ['Lucknow', 'LUCKNOW']
Possible city variation: ['Howrah', 'HOWRAH']
Possible city variation: ['Kolkata', 'KOLKATA']


In [15]:

# Create a separate standardized city column
clean_df["City_Unified"] = clean_df["City"].str.strip()

# Correct verified capitalization variations
city_mapping = {
    "YERPEDU": "Yerpedu",
    "BILASPUR (Chhattisgarh)": "Bilaspur (Chhattisgarh)",
    "NEW DELHI": "New Delhi",
    "SURAT": "Surat",
    "NAGPUR": "Nagpur",
    "BHUBANESWAR": "Bhubaneswar",
    "PUDUCHERRY": "Puducherry",
    "HYDERABAD": "Hyderabad",
    "LUCKNOW": "Lucknow",
    "HOWRAH": "Howrah",
    "KOLKATA": "Kolkata"
}

clean_df["City_Unified"] = clean_df["City_Unified"].replace(city_mapping)

# Validate the changes
print("Original unique cities:", clean_df["City"].nunique())

print("Standardized unique cities:",
      clean_df["City_Unified"].nunique())

# Display corrected records
display(
    clean_df[
        clean_df["City"].ne(clean_df["City_Unified"])
    ][["Year", "Name", "City", "City_Unified", "State_Unified"]]
)


Original unique cities: 378
Standardized unique cities: 367


,Year,Name,City,City_Unified,State_Unified
2348,2020,National Institute of Pharmaceutical Education...,LUCKNOW,Lucknow,Uttar Pradesh
2378,2020,Smt. Kishoritai Bhoyar College of Pharmacy,NAGPUR,Nagpur,Maharashtra
3635,2022,Institute of Home Economics,NEW DELHI,New Delhi,Delhi
3750,2022,Indian Institute of Engineering Science and Te...,HOWRAH,Howrah,West Bengal
3752,2022,Kalinga Institute of Industrial Technology,BHUBANESWAR,Bhubaneswar,Odisha
3766,2022,"Indian Institute of Technology, Tirupati",YERPEDU,Yerpedu,Andhra Pradesh
5379,2024,Kanchi Mamunivar Govt. Institute for Postgradu...,PUDUCHERRY,Puducherry,Pondicherry
5511,2024,Sardar Vallabhbhai National Institute of Techn...,SURAT,Surat,Gujarat
5513,2024,Indian Institute of Technology Tirupati,YERPEDU,Yerpedu,Andhra Pradesh
5706,2024,Guru Ghasidas Vishwavidyalaya,BILASPUR (Chhattisgarh),Bilaspur (Chhattisgarh),Chhattisgarh


In [16]:

# Check for remaining capitalization and spacing variations
city_check = (
    clean_df.dropna(subset=["City_Unified", "State_Unified"])
    .groupby([
        clean_df["State_Unified"].str.strip().str.lower(),
        clean_df["City_Unified"].str.strip().str.lower()
    ])["City_Unified"]
    .unique()
)

remaining_variations = []

for key, values in city_check.items():
    if len(values) > 1:
        remaining_variations.append(list(values))

print("Remaining possible city variations:",
      len(remaining_variations))

for variation in remaining_variations:
    print(variation)


Remaining possible city variations: 0


In [17]:
# Check the number of unique institution names
print("Unique institutions:",
      df["Name"].nunique())

# Check total records
print("Total records:",
      len(df))

# Display frequently occurring institution names
print("\nMost frequently occurring names:")

print(
    df["Name"]
    .value_counts()
    .head(20)
)

Unique institutions: 1334
Total records: 6966

Most frequently occurring names:
Name
Aligarh Muslim University                               65
Kalinga Institute of Industrial Technology              61
Banaras Hindu University                                53
Panjab University                                       53
Anna University                                         52
Lovely Professional University                          51
Saveetha Institute of Medical and Technical Sciences    51
Siksha `O` Anusandhan                                   48
Jamia Millia Islamia                                    48
Amrita Vishwa Vidyapeetham                              47
Indian Institute of Technology Kharagpur                44
Birla Institute of Technology                           44
Amity University                                        43
Institute of Chemical Technology                        41
Jadavpur University                                     39
Jamia Hamdard                 

In [18]:
# Create a temporary normalized name
df["Name_Check"] = (
    df["Name"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

# Count distinct original names under each
# normalized name
name_variations = (
    df.groupby("Name_Check")["Name"]
    .nunique()
    .reset_index(name="Variation_Count")
)

# Keep names having multiple variations
name_variations = name_variations[
    name_variations["Variation_Count"] > 1
]

print("Potential name variations:",
      len(name_variations))

Potential name variations: 33


In [19]:
# Display the names that share the same normalized name
for key in name_variations["Name_Check"]:
    print("\n---", key, "---")

    matching_names = df.loc[
        df["Name_Check"] == key, "Name"
    ].drop_duplicates()

    for name in matching_names:
        print(name)


--- amity university ---
AMITY University
Amity University

--- army institute of law ---
Army Institute of  Law
Army Institute of Law

--- birla institute of technology & science -pilani ---
Birla Institute of Technology & Science -Pilani
Birla Institute of Technology  & Science -Pilani

--- chitkara university ---
Chitkara University
Chitkara  University

--- cvr college of engineering ---
CVR College of Engineering
CVR College Of Engineering

--- dr d y patil institute of pharmaceutical sciences and research ---
DR D Y Patil Institute of Pharmaceutical Sciences and Research
Dr D Y Patil Institute of Pharmaceutical Sciences and Research

--- fore school of management ---
Fore School of Management
FORE School of Management

--- g. h. raisoni college of engineering, nagpur ---
G. H. Raisoni College of Engineering, NAGPUR
G. H. Raisoni College of Engineering, Nagpur

--- gandhi institute of technology and management (gitam) ---
Gandhi Institute of Technology and Management (GITAM)
Gand